In [68]:
import wrds
import pandas as pd
import os
from sqlalchemy import text

In [ ]:
START = "2011-01-01"
END   = "2025-12-31"

TICKERS = {
    "AAPL":  "AAPL",
    "MSFT":  "MSFT",
    "NVDA":  "NVDA",   
    "AMZN":  "AMZN",
    "JPM":   "JPM",
    "JNJ":   "JNJ",
    "XOM":   "XOM",
    "TSLA":  "TSLA",
    "NFLX":  "NFLX",
    "V":     "V",
}
ticker_list = tuple(TICKERS.values())

In [ ]:
# Connect to WRDS
db = wrds.Connection()  

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [71]:
query = f"""
    SELECT b.ticker, a.date,
        ABS(a.prc) AS close, a.openprc AS open,
        a.askhi AS high, a.bidlo AS low,
        a.vol AS volume, a.ret AS return
    FROM crsp.dsf AS a
    JOIN crsp.dsenames AS b
        ON  a.permno = b.permno
        AND a.date  >= b.namedt
        AND a.date  <= COALESCE(b.nameendt, CURRENT_DATE)
    WHERE b.ticker IN {ticker_list}
      AND a.date BETWEEN '{START}' AND '{END}'
    ORDER BY b.ticker, a.date
"""

with db.engine.connect() as conn:
    df_raw = pd.read_sql(text(query), conn, parse_dates=["date"])

print(f"Downloaded {len(df_raw):,} rows across {df_raw['ticker'].nunique()} tickers.")
df_raw.head()

Downloaded 35,220 rows across 10 tickers.


,ticker,date,close,open,high,low,volume,return
0,AAPL,2011-01-03,329.57001,325.64001,330.26001,324.83649,16438280.0,0.021732
1,AAPL,2011-01-04,331.29001,332.44000,332.50000,328.14999,11444072.0,0.005219
2,AAPL,2011-01-05,334.00000,329.54999,334.34000,329.50000,9486655.0,0.008180
3,AAPL,2011-01-06,333.73001,334.71899,335.25000,332.89999,11138250.0,-0.000808
4,AAPL,2011-01-07,336.12000,333.98999,336.35001,331.89999,11645048.0,0.007161


In [72]:
# Check which tickers actually came back
print("Tickers in result:", sorted(df_raw["ticker"].unique()))

# Split into per-ticker DataFrames
raw = {}
for name, crsp_ticker in TICKERS.items():
    sub = df_raw[df_raw["ticker"] == crsp_ticker].copy()
    if sub.empty:
        print(f"WARNING: {name} ({crsp_ticker}) returned no data — check ticker spelling in CRSP")
        continue
    sub = sub.set_index("date").drop(columns="ticker")
    sub.index = pd.to_datetime(sub.index)
    raw[name] = sub
    print(f"{name:6s}: {len(sub):4d} rows  |  {sub.index[0].date()} -> {sub.index[-1].date()}  |  NAs: {sub.isna().sum().sum()}")

Tickers in result: ['AAPL', 'AMZN', 'JNJ', 'JPM', 'MSFT', 'NFLX', 'NVDA', 'TSLA', 'V', 'XOM']
AAPL  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
MSFT  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
NVDA  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
AMZN  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
JPM   : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
JNJ   : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
XOM   : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
TSLA  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
NFLX  : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0
V     : 3522 rows  |  2011-01-03 -> 2024-12-31  |  NAs: 0


In [73]:
# Summary statistics on close prices
for name, df in raw.items():
    print(f"\n--- {name} ---")
    display(df["close"].describe())


--- AAPL ---


count    3522.000000
mean      243.929960
std       152.604100
min        90.280000
25%       134.164997
50%       175.620005
75%       336.050005
max       702.099980
Name: close, dtype: float64


--- MSFT ---


count    3522.000000
mean      143.431191
std       125.309334
min        23.705000
25%        41.480000
50%        86.150000
75%       243.022500
max       467.560000
Name: close, dtype: float64


--- NVDA ---


...

count    3522.00000
mean      313.12258
std       282.79610
min        21.83000
25%       177.61000
50%       237.73500
75%       338.66500
max      2238.75000
Name: close, dtype: float64


--- NFLX ---


count    3522.000000
mean      322.896964
std       180.236659
min        53.800000
25%       173.032495
50%       320.789995
75%       441.013748
max       936.560000
Name: close, dtype: float64


--- V ---


count    3522.000000
mean      166.845153
std        65.499045
min        64.520000
25%       102.952500
50%       177.505000
75%       216.435000
max       320.910000
Name: close, dtype: float64

In [74]:
os.makedirs("../data/stocks", exist_ok=True)

for name, df in raw.items():
    out = f"../data/stocks/raw_{name.lower()}.csv"
    df.to_csv(out)
    print(f"Saved {out}")

db.close()

Saved ../data/stocks/raw_aapl.csv
Saved ../data/stocks/raw_msft.csv
Saved ../data/stocks/raw_nvda.csv
Saved ../data/stocks/raw_amzn.csv
Saved ../data/stocks/raw_jpm.csv
Saved ../data/stocks/raw_jnj.csv
Saved ../data/stocks/raw_xom.csv
Saved ../data/stocks/raw_tsla.csv
Saved ../data/stocks/raw_nflx.csv
Saved ../data/stocks/raw_v.csv
